In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from scipy import stats
from scipy.stats import gaussian_kde

matplotlib.rcParams["font.family"] = "DejaVu Sans"

DISTRIBUTIONS = {
    "normal": {
        "name_ru": "Нормальное N(0,1)",
        "sample":  lambda n: np.random.normal(0.0, 1.0, n),
        "cdf":     lambda x: stats.norm.cdf(x, 0, 1),
        "pdf":     lambda x: stats.norm.pdf(x, 0, 1),
        "x_range": np.linspace(-4, 4, 500),
        "discrete": False,
    },
    "cauchy": {
        "name_ru": "Коши C(0,1)",
        "sample":  lambda n: stats.cauchy.rvs(loc=0.0, scale=1.0, size=n),
        "cdf":     lambda x: stats.cauchy.cdf(x, 0, 1),
        "pdf":     lambda x: stats.cauchy.pdf(x, 0, 1),
        "x_range": np.linspace(-4, 4, 500),
        "discrete": False,
    },
    "laplace": {
        "name_ru": "Лапласа L(0,1/√2)",
        "sample":  lambda n: np.random.laplace(0.0, 1.0 / np.sqrt(2), n),
        "cdf":     lambda x: stats.laplace.cdf(x, 0, 1.0 / np.sqrt(2)),
        "pdf":     lambda x: stats.laplace.pdf(x, 0, 1.0 / np.sqrt(2)),
        "x_range": np.linspace(-4, 4, 500),
        "discrete": False,
    },
    "poisson": {
        "name_ru": "Пуассона P(10)",
        "sample":  lambda n: np.random.poisson(10, n),
        "cdf":     lambda x: stats.poisson.cdf(x, 10),
        "pdf":     lambda k: stats.poisson.pmf(k, 10),
        "x_range": np.linspace(6, 14, 500),
        "x_range_pdf": np.arange(6, 15),
        "discrete": True,
    },
    "uniform": {
        "name_ru": "Равномерное U(-√3,√3)",
        "sample":  lambda n: np.random.uniform(-np.sqrt(3), np.sqrt(3), n),
        "cdf":     lambda x: stats.uniform.cdf(x, -np.sqrt(3), 2 * np.sqrt(3)),
        "pdf":     lambda x: stats.uniform.pdf(x, -np.sqrt(3), 2 * np.sqrt(3)),
        "x_range": np.linspace(-4, 4, 500),
        "discrete": False,
    },
}

SIZES = [20, 60, 100]

def empirical_cdf(sample: np.ndarray, x: np.ndarray) -> np.ndarray:
    sample_sorted = np.sort(sample)
    return np.searchsorted(sample_sorted, x, side="right") / len(sample)

def plot_cdf(dist_key: str, d: dict):
    x = d["x_range"]
    fig, axes = plt.subplots(1, len(SIZES), figsize=(13, 4))
    fig.suptitle(
        f"ЭФР и теоретическая ФР\nРаспределение: {d['name_ru']}",
        fontsize=12, weight="bold", y=1.02
    )

    for ax, n in zip(axes, SIZES):
        sample = d["sample"](n)

        ax.plot(x, d["cdf"](x), color="red", linewidth=2,
                label="Теоретическая ФР", zorder=3)

        ecdf_vals = empirical_cdf(sample, x)
        ax.step(x, ecdf_vals, color="steelblue", linewidth=1.5,
                where="post", label="ЭФР", zorder=2)

        ax.set_title(f"n = {n}", fontsize=10)
        ax.set_xlabel("x", fontsize=9)
        ax.set_ylabel("F(x)", fontsize=9)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3, linestyle="--")
        ax.set_ylim(-0.05, 1.05)

    plt.tight_layout()
    fname = f"lab4_cdf_{dist_key}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Сохранён: {fname}")

def plot_kde(dist_key: str, d: dict):
    x = d.get("x_range")
    x_pdf = d.get("x_range_pdf", x)
    fig, axes = plt.subplots(1, len(SIZES), figsize=(13, 4))
    fig.suptitle(
        f"KDE и теоретическая плотность\nРаспределение: {d['name_ru']}",
        fontsize=12, weight="bold", y=1.02
    )

    for ax, n in zip(axes, SIZES):
        sample = d["sample"](n)

        if d["discrete"]:
            ax.vlines(x_pdf, 0, d["pdf"](x_pdf),
                      color="red", linewidth=1.5, label="Теоретическая PMF")
            ax.plot(x_pdf, d["pdf"](x_pdf), "ro", markersize=4)
        else:
            ax.plot(x, d["pdf"](x), color="red", linewidth=2,
                    label="Теоретическая плотность", zorder=3)

        ax.hist(sample, bins="auto", density=True,
                alpha=0.35, color="gray", edgecolor="white",
                linewidth=0.3, label="Гистограмма")

        if not d["discrete"] or n >= 20:
            kde = gaussian_kde(sample, bw_method="silverman")
            ax.plot(x, kde(x), color="steelblue", linewidth=2,
                    label="KDE (Гаусс)", zorder=4)

        ax.set_title(f"n = {n}", fontsize=10)
        ax.set_xlabel("x", fontsize=9)
        ax.set_ylabel("Плотность", fontsize=9)
        ax.legend(fontsize=7)
        ax.grid(alpha=0.3, linestyle="--")

    plt.tight_layout()
    fname = f"lab4_kde_{dist_key}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Сохранён: {fname}")



def main():

    for dist_key, d in DISTRIBUTIONS.items():
        print(f"\n{d['name_ru']}")
        plot_cdf(dist_key, d)
        plot_kde(dist_key, d)


if __name__ == "__main__":
    main()
